랭체인에서 체인이란?
Prompt와 LLM이 결합된 구조를 말한다.
사용자입력 (프롬프트)를 받아 LLM으로 응답을 생성하는 구조를 말한다.

추가 실습내용은 다음과 같다.
1. 멀티체인구성
2. Runnable 프로토콜을 구현한 객체를 조금 더 알아보자.

API발급받은 후 테스트 해보자.

In [1]:
!pip install -q langchain langchain-openai

In [2]:
# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")

import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


OpenAI API Key:  ········


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

result = llm.invoke("랭체인이란?")
print(result.content)

랭체인(Chain of Thought, COT)은 자연어 처리(NLP) 분야에서 모델이 문제를 해결하거나 질문에 대답할 때 단계별로 사고 과정을 나타내는 방법론을 의미합니다. 이 접근법은 모델이 단순히 결과를 제시하는 것이 아니라, 그 과정에서 어떤 논리적 추론을 거쳤는지를 보여주는 것을 촉진합니다.

랭체인의 주요 목적은 다음과 같습니다:

1. **투명성 제공**: 모델의 결정 과정이 명확해지면, 사용자나 연구자가 결과를 이해하고 분석하는 데 도움이 됩니다.
2. **문제 해결 능력 향상**: 문제를 해결하는 과정에서 중간 단계의 결과를 활용하여 더 복잡한 문제를 다룰 수 있습니다.
3. **오류 분석**: 모델이 잘못된 답변을 하는 경우, 그 과정에서 어떤 오류가 발생했는지를 파악할 수 있어 개선 점을 찾는 데 유용합니다.

랭체인은 주로 고급 언어 모델과 결합되어 사용되며, 특히 복잡한 질문에 대한 답변, 수학 문제 해결, 또는 프로그래밍 코드 생성 등 다양한 측면에서 효과적입니다.


이제 단일체인을 구성해 보자 <br>
아까는 Prompt + Model 이였다면 이번에는
Prompt + Model + parser의 형태로 단일 체인을 만들어 볼 것이다.<br>

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatPromptTemplate.from_messages은 [ ]안에 역할지정 및 여러 메시지를 넣고 싶을때 사용한다.

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an computerscience teacher in middel school."),
    ("user", "Answer the question: {input}")
])

llm = ChatOpenAI(model="gpt-4o-mini")

# LLM의 복잡한 응답객체에서 우리가 원하는 깔끔한 문자열만 받을수 있게 도와주는 메서드
output_parser = StrOutputParser()

# LCEL chaining ((LangChain Expression Language) 문법)
# LangChain에서는 특별히 앞 단계의 출력을 다음 단계의 입력으로 연결하는 의미로 오버로딩(overload) 되어있음
chain = prompt | llm | output_parser

# invoke - 한번 실행후 결과를 반환하는 함수 (실행함수)
# chain 호출 - dict가 인자의 자료형임
chain.invoke({"input": "랭체인이란?"})

'랭체인(LLM Chain)은 대형 언어 모델(LLM: Large Language Model)을 활용하여 다양한 작업을 수행하고, 이들 작업을 서로 연결하여 복잡한 프로세스를 자동화하는 시스템입니다. 기본적으로 랭체인은 여러 개의 순차적 또는 병렬 작업을 구성하여, 사용자가 원하는 결과를 효과적으로 도출할 수 있도록 돕습니다.\n\n예를 들어, 랭체인을 사용하면 다음과 같은 작업을 수행할 수 있습니다:\n\n1. **질문 분석**: 사용자의 질문을 이해하고 분석합니다.\n2. **정보 검색**: 관련 정보를 데이터베이스나 외부 소스에서 검색합니다.\n3. **응답 생성**: 검색된 정보를 바탕으로 자연스러운 언어로 응답을 생성합니다.\n4. **후처리**: 생성된 응답을 검토하고 필요에 따라 수정합니다.\n\n랭체인은 특히 챗봇, 고객 지원 시스템, 콘텐츠 생성 등 다양한 분야에서 응용될 수 있습니다. 이를 통해 사용자는 보다 효율적이고 일관된 경험을 제공받을 수 있습니다.'

지금까지 본 것이 단일체인이라면<br>
지금 부터는 멀티체인을 보자.<br>
멀티체인은 말 그래도 여러개의 체인을 연결하는 구조를 말한다.

세개의 체인을 연결해보자. <br>
chain1 : "LLM 기초" -> 객관식 문제 3문제 생성 <br>
chain2 : 문제를 입력받아 정답과 해설을 생성 <br>
chain3 : 정답/해설을 분석해 문제의 난이도를 분류 <br>

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# prompt 정의
# 역할지정 안하고 단일메시지 보낼때는 from_message 말고 from_template 사용하면 됨

prompt = ChatPromptTemplate.from_template(
    """너는 고등학교 선생님이야.
    주제:{topic}
    위 주제에 대한 4지선다 객관식 문제를 3개 만들어줘
    문제:
    보기:
    """
)

prompt2 = ChatPromptTemplate.from_template(
    """
    너는 출판사의 해설작성 전문가야
    다음 문제들의 정답과 해설을 작성해줘
    {problems}
    정답:
    해설:
    """
)

prompt3 = ChatPromptTemplate.from_template(
    """
    다음 문제와 해설을 읽고 난이도를 분류해 주세요
    분류: 쉬움, 보통, 어려움
    {answer}
    """
)


llm = ChatOpenAI(model="gpt-4o-mini")

topic ="AI 이론"

# chain1 = prompt | llm | StrOutputParser()
# chain2 = prompt2 | llm | StrOutputParser()
# chain3= prompt3 | llm | StrOutputParser()

# # 문제생성
# problem = chain1.invoke({"topic": topic})
# print(f'생성된문제 {problem}')

# # 정답 및 해설
# answer = chain2.invoke({"problems": problem})
# print(f'정답 및 해설 {answer}')

# # 난이도분류
# difficulty = chain3.invoke({"answer": answer})
# print(f'난이도분류 {difficulty}')


# 코딩 컨벤션을 인라인 형식으로도 가능 (예제)
problem = (prompt | llm | StrOutputParser()).invoke({"topic": topic})
answer = (prompt2 | llm | StrOutputParser()).invoke({"problems": problem})
difficulty = (prompt3 | llm | StrOutputParser()).invoke({"answer": answer})

print(answer)
print("=============================")
print(difficulty)


문제 1: 인공지능의 기본적인 구성 요소 중 하나로, 데이터에서 패턴을 학습하고 예측을 수행하는 알고리즘을 무엇이라 하는가?

가) 데이터베이스  
나) 기계 학습  
다) 클라우드 컴퓨팅  
라) 네트워크  

**정답: 나) 기계 학습**

**해설:**  
기계 학습(Machine Learning)은 인공지능의 한 분야로서, 데이터를 기반으로 패턴을 찾아내고 이를 바탕으로 예측을 수행하는 알고리즘들을 포함합니다. 기계 학습은 데이터에서 자동으로 학습해 나가는 과정이므로, 학습에 사용된 데이터가 많을수록 예측의 정확성이 높아질 수 있습니다. 반면, 데이터베이스는 데이터를 저장하는 시스템, 클라우드 컴퓨팅은 인터넷을 통해 데이터 처리 및 저장을 수행하는 기술, 네트워크는 데이터 통신과 연결을 위한 시스템을 의미합니다. 

---

문제 2: 다음 중 딥러닝의 주된 원리로 올바른 것은 무엇인가?

가) 심층 신경망 구조를 통해 다양한 데이터의 특징을 자동으로 추출한다.  
나) 데이터를 고정된 규칙에 따라 분류한다.  
다) 모든 데이터를 수동으로 입력하여 처리한다.  
라) 신경망이 아닌 전통적인 알고리즘을 사용한다.  

**정답: 가) 심층 신경망 구조를 통해 다양한 데이터의 특징을 자동으로 추출한다.**

**해설:**  
딥러닝(Deep Learning)은 인공 신경망의 심화된 형태로, 다층의 신경망을 통해 데이터를 처리합니다. 이러한 구조는 여러 층을 통해 다양한 수준의 특징을 자동으로 추출할 수 있어, 이미지 인식, 음성 인식 등의 분야에서 탁월한 성능을 발휘합니다. 이에 비해 고정된 규칙에 따른 분류나 모든 데이터를 수동 입력하여 처리하는 방식은 전통적인 기계 학습 방법에 해당하며, 신경망을 사용하지 않는 전통적인 알고리즘과는 차별화된 접근법입니다.

---

문제 3: 자연어 처리를 이루는 과정 중에서, 문장을 토큰화하고 단어의 의미를 이해하기 위해 사용하는 임베딩 기술을 무엇이라 하는가?

가) 이미지 리사이징  
나) 워드 임베딩  
다) 

* "프롬프트+LLM+파서”가 각각의 단위의 모듈이고 (Runnable 컴포넌트)<br> 

* invoke는 실행/연결 지점이라고 생각하면 가장 이해하기 쉽다. <br>

* 각 체인은 독립적으로 정의되어야 한다. 그리고 invoke 호출 시 이전 체인의 출력값을 입력으로 넣으면 된다. <br>

* 이렇게 하면 단계별로 중간 결과 확인이 가능 + 디버깅에 용이하다


LangChain을 호출하고 결과를 반환 할때 invoke 메소드를 사용했다.<br>
이러한 invoke 메소드를 "Runnable" 프로토콜 이라고 부른다. <br>
"Runnable" 프로토콜로는 invoke, batch, ainvoke 실행 메소드 들이있다 <br>
* batch는 입력 리스트에 대해 체인을 호출하고 결과를 리스트로 반환한다 <br>
* stream은 모델이 토큰을 생성하는 즉시 바로바로 흘려보내준다. 따라서 chatbot처럼 한글자씩 타이핑 되는 느낌을 주고 싶을때 사용하거나 긴 텍스트 요약을 토큰반위로 받아서 실시간으로 ui에 띄울떄 사용한다 <br>
* ainvoke는 단일 입력 실행을 비동기(async/await) 방식으로 지원

    * batch -> 비동기로 여러 입력을 동시에 실행, 입력 순서대로 결과를 한번에 반환
    * ainvoke -> 비동기 실행 (결과는 한 번에 반환)
    * astream ->  비동기 스트리밍 실행 (결과를 청크 단위로 흘려줌)



In [8]:
# batch 메소드 사용예시

# 1. 컴포넌트 정의
prompt= ChatPromptTemplate.from_template("중학생이 이해하도록 {topic}에 대해서 간단하게 설명해주세요")

model = ChatOpenAI(model="gpt-4o-mini")

output_parser = StrOutputParser()

# 2. 컴포넌트 연결
chain = prompt | model | output_parser

# 3. batch 메소드 사용
topic = ["랭체인","RAG","Fine-tuning"]
result= chain.batch(topic)
for i in range(len(topic)):
    print(f'{topic[i]}에 대한 설명:{result[i]}')
    print(f'*****************')


랭체인에 대한 설명:랭체인(Chain of Thought)이라는 개념은 주로 인공지능(AI) 모델이 문제를 해결하거나 질문에 답할 때 사용하는 방법 중 하나입니다. 쉽게 말해, 단계별로 생각하는 과정을 표현하는 방법이라고 할 수 있어요.

예를 들어, 어떤 수학 문제를 해결할 때 그냥 정답만 말하는 것이 아니라, 문제를 이해하고, 어떤 공식을 쓸지 고민하고, 계산을 하는 과정을 차례로 설명하는 것이죠. 이렇게 단계적으로 생각을 정리하면, 더 복잡한 문제도 잘 해결할 수 있고, 잘못된 답을 줄 확률도 줄어듭니다.

랭체인은 이러한 생각의 과정을 글로 표현함으로써 인공지능이 인간처럼 논리적이고 체계적으로 사고할 수 있게 도와줍니다. 그래서 더 나은 답변을 만들 수 있도록 하는 기술입니다.
*****************
RAG에 대한 설명:RAG는 "Retrieval-Augmented Generation"의 약자로, 정보를 검색하고 생성하는 두 가지 과정을 결합한 방법이에요. 쉽게 말하면, 우리가 질문을 할 때 그에 대한 답변을 찾기 위해 필요한 정보(예: 책, 웹사이트 등)를 먼저 검색하고, 그 정보를 바탕으로 새로운 문장을 만들거나 답변을 구성하는 방식이에요.

예를 들어, "태양계의 행성에 대해 알려줘"라는 질문을 하면, RAG는 관련된 정보를 여러 곳에서 찾고, 그 정보를 참고하여 "태양계에는 지구, 화성, 목성 등 여러 행성이 있어요"라는 문장을 만들어 내는 거죠. 이렇게 하면 더 정확하고 풍부한 정보를 제공할 수 있어요. 

결론적으로, RAG는 정보를 검색하고 그것을 기반으로 새로운 내용을 만들어내는 똑똑한 방법이라고 할 수 있어요!
*****************
Fine-tuning에 대한 설명:Fine-tuning은 간단히 말해서, 미리 학습된 모델을 특정한 작업이나 데이터에 맞게 조금 더 학습시키는 과정이에요. 

예를 들어, 이미 많은 종류의 그림을 인식할 수 있도록 훈련된 로봇이 있다고 생각해보세요. 이 로봇은 고양이와 강아지를 잘 알아보지

In [9]:
# stream 메소드 사용 예시

stream=chain.stream({"topic":"허깅페이스"}) # 청크(단어뭉치) 단위로 쪼개서 출력

for i in stream:
  print(i,end='',flush=True) # 버퍼를 비워 출력을 바로 화면에 띄움


허깅페이스(Hugging Face)는 인공지능(AI) 기술, 특히 자연어 처리(NLP) 분야에서 유명한 회사입니다. 이 회사는 사람들이 쉽고 빠르게 AI 모델을 사용할 수 있도록 도와주는 다양한 도구와 라이브러리를 제공합니다.

1. **AI 모델**: 허깅페이스는 텍스트를 이해하고 생성하는 데 도움을 주는 여러 AI 모델을 만들어 제공합니다. 예를 들어, 질문에 답하거나, 글을 요약하거나, 대화를 할 수 있는 모델들이 있습니다.

2. **트랜스포머**: 허깅페이스의 가장 유명한 제품 중 하나는 '트랜스포머(Transformers)'라는 라이브러리입니다. 이 라이브러리는 많은 종류의 AI 모델을 쉽게 사용할 수 있게 해줍니다. 프로그래밍이 조금만 가능하면, 누구나 이 모델들을 쉽게 사용할 수 있어요.

3. **커뮤니티**: 허깅페이스는 많은 사용자들이 서로 정보를 나누고, 자신이 만든 AI 모델을 공유할 수 있는 플랫폼을 제공합니다. 이렇게 하면 누구나 더 나은 AI를 만드는 데 기여할 수 있습니다.

4. **교육 자료**: 초보자도 쉽게 AI를 이해할 수 있도록 많은 교육 자료와 튜토리얼을 제공합니다.

요약하자면, 허깅페이스는 사람들이 자연어 처리 AI를 더 쉽게 사용할 수 있도록 도와주는 도구와 자원을 제공하는 회사입니다.

In [10]:
# ainvoke 예시

import asyncio                         # 파이썬 내장 비동기 라이브러리
from asyncio.tasks import as_completed # 비동기 작업을 "완료되는 순서대로" 처리하기 위한 함수
import nest_asyncio                    # 코랩, 주피터노트북에서 비동기 작업시 추가적으로 필요

nest_asyncio.apply()                   # (코랩/주피터에서만 필요, 비동기 실행 전에 1번만 선언)

async def run_async():                 # 비동기 함수 선언

    task=[
        chain.ainvoke({"topic":"랭체인"}),
        chain.ainvoke({"topic":"RAG"}),
        chain.ainvoke({"topic":"Fine-tuning"})
    ]

    for i in asyncio.as_completed(task): # 작업이 끝나는 순서대로 반환
        print(await i)                   # 비동기 작업 하나씩 await해서 결과로 출력
        print(f'*****************')

asyncio.run(run_async())  # 비동기 함수 실행할 때

RAG는 "Retrieval-Augmented Generation"의 약자입니다. 쉽게 말해, 정보를 검색해서 더 좋은 답변을 만들어 내는 방법을 말해요. 

예를 들어, 우리가 질문을 하면, RAG는 그 질문에 맞는 정보를 인터넷이나 데이터베이스에서 찾아냅니다. 그리고 찾은 정보를 바탕으로 더 정확하고 풍부한 답변을 생성해주는 방식입니다. 

즉, RAG는 단순히 이미 알고 있는 것만으로 답변하는 것이 아니라, 필요할 때 필요한 정보를 찾아서 더 똑똑한 답변을 만드는 기술이에요. 이렇게 하면 우리가 원하는 정보를 더 잘 찾고, 이해하는 데 도움이 됩니다.
*****************
Fine-tuning은 머신 러닝, 특히 인공지능 모델을 더 잘 가르치기 위해 사용하는 방법이에요. 간단히 말해서, 이미 훈련된 모델을 바탕으로 특정한 작업이나 데이터를 위해 조금 더 학습시키는 과정을 의미해요.

예를 들어, 우리가 강아지를 인식하는 모델을 만들었다고 가정해볼게요. 이 모델은 다양한 강아지 사진을 보고 강아지를 인식하는 방법을 배웠어요. 하지만 이제 우리는 모델이 특정한 종류의 강아지, 예를 들어 ‘푸들’을 더 잘 인식하도록 하고 싶어요. 이럴 때 기존에 배운 것을 바탕으로 푸들 사진 몇 장을 더 보여주고 다시 학습시키는 거죠. 이렇게 하면 모델이 푸들을 더 잘 인식할 수 있게 되는 거예요.

Fine-tuning은 마치 학생이 수업을 듣고 기본 개념을 이해한 다음, 추가로 특정 주제에 대해 더 깊이 공부하는 것과 비슷하답니다.
*****************
랭체인(Language Chain)은 인공지능이 언어를 이해하고 생성하는 데 도움을 주는 도구와 기술의 모음이에요. 쉽게 말해서, 컴퓨터가 우리처럼 자연스럽게 대화하거나 글을 쓸 수 있도록 하는 방법이죠.

랭체인의 핵심은 '체인'이라는 개념이에요. 말 그대로 여러 단계로 연결된 과정을 통해 정보를 처리하는 방식입니다. 예를 들어, 질문을 받으면 첫 번째 단계에서 그 질문을 이해하고, 두 번째 단계에서

여러번 실행해 보면 응답받은 순서대로 출력이 될 것입니다.

이제 다시 돌아가서 Rag에 대해서 살펴보겠습니다.